In [73]:
import sys
import weakref
import gc

## Задание 1

Сформулировать класс, который демонстрирует, как CPython хранит данные экземпляра в словаре и как это связано с атрибутом `__dict__`.

Реализуйте класс TrackedObject, который:
- В конструкторе принимает произвольные именованные аргументы и записывает их в атрибуты экземпляра.
- Переопределяет `__setattr__` и `__delattr__`, чтобы:
  - Логировать каждое изменение в списке history (атрибут экземпляра).
  - Отслеживать реальный размер `__dict__` до и после операции.

In [74]:
class TrackedObject:
    def __init__(self, **kwargs):
        # history сам добавляем через super().__setattr__, чтобы не ловить в логе
        super().__setattr__("history", [])
        for key, value in kwargs.items():
            self.__setattr__(key, value)

    def __setattr__(self, name, value):
        super().__setattr__(name, value)
        self.history.append(
            f"Value of {name} setted to {value} \n {len(self.__dict__) = }"
        )

    def __delattr__(self, name):
        del self.__dict__[name]
        self.history.append(f"Deleted {name} :( \n {len(self.__dict__) = }")


obj = TrackedObject(x=1, y=2)
obj.z = 3  # добавление нового атрибута
obj.__str__ = lambda s: "patched"  # перезатирка метода
del obj.x

print(obj.__dict__.keys())
for event in obj.history:
    print(event)

dict_keys(['history', 'y', 'z', '__str__'])
Value of x setted to 1 
 len(self.__dict__) = 2
Value of y setted to 2 
 len(self.__dict__) = 3
Value of z setted to 3 
 len(self.__dict__) = 4
Value of __str__ setted to <function <lambda> at 0x0000024FFAE03E20> 
 len(self.__dict__) = 5
Deleted x :( 
 len(self.__dict__) = 4


## Задание 2

Создать нетривиальный ромбовидный и более сложный граф наследования, а затем вручную вывести C3‑линеаризацию и сверить с `__mro__`:

- Постройте иерархию классов не менее чем из 6 классов с несколькими ромбами (несколько общих предков).
- В одной из веток сделайте «конфликт» имён методов (одинаковый метод в двух разных базах).
- Напишите функцию `c3_linearize(cls)`, которая по списку баз реализует алгоритм C3‑линеаризации (без использования внутренностей CPython).
- Для нескольких классов:
  - Выведите результат вашей функции.
  - Выведите `cls.__mro__`.
- Прокомментируйте, почему порядок разрешения методов именно такой, и как C3 гарантирует локальный порядок и отсутствие конфликтов.

In [75]:
def merge(seqs):
    res = []
    while True:
        non_empty = [s for s in seqs if s]
        if not non_empty:
            return res

        for seq in non_empty:
            candidate = seq[0]
            # Check if candidate appears in the tail of any other sequence
            tails = [s[1:] for s in non_empty if len(s) > 1]
            if all(candidate not in tail for tail in tails):
                res.append(candidate)
                # Remove candidate from all sequences
                for s in seqs:
                    if s and s[0] == candidate:
                        s.pop(0)
                break
        else:
            # No candidate found -> inconsistent hierarchy (shouldn't happen in valid Python classes)
            raise TypeError("Inconsistent hierarchy, no C3 MRO is possible")


def c3_linearize(cls):
    if not cls.__bases__:
        return [cls]

    # Get linearizations of parents
    parent_seqs = [c3_linearize(parent) for parent in cls.__bases__]
    # Add the list of direct parents as the last sequence
    parent_seqs.append(list(cls.__bases__))

    return [cls] + merge(parent_seqs)


# Your code here (e.g. classes A, B, C, D ...)


class A: ...


class B(A): ...


class C(A): ...


class D(B, C): ...  # Diamond 1: D -> B, C -> A


class E(C): ...


class F(D, E):  # Diamond 2: F -> D, E -> C (Complex overlap)
    def solve(self):
        return "F"


class G(B):
    def solve(self):
        return "G"


class H(F, G): ...  # Final complex node: H -> F, G -> B


test_cls = H

print(f"\n=== Class: {test_cls.__name__} ===")

# Your code here to show results of c3_linearize and cls.__mro__

# 1. Custom C3 Implementation
try:
    custom_mro = c3_linearize(test_cls)
    print(f"Custom C3: {[c.__name__ for c in custom_mro]}")
except TypeError as e:
    print(f"Custom C3 Error: {e}")

# 2. Python's Native MRO
native_mro = test_cls.__mro__
print(f"Native MRO:  {[c.__name__ for c in native_mro]}")

# 4. Method Resolution Demo
instance = H()
print(f"Method 'solve' resolved to: {instance.solve()}")

# Local Precedence: In 'class H(F, G)', F always precedes G. In 'class D(B, C)', B precedes C.

# Monotonicity: If B precedes A in class B's MRO, B must precede A in H's MRO. C3 guarantees this.

# Conflict Resolution: In 'H(F, G)', F is listed first.
# Therefore, its 'solve' method overrides G's. C3 ensures the left-to-right order is respected
# while maintaining consistency with the parents' own internal orders.

# Diamond Handling: For 'H', 'A' appears only once, after all its children (B, C, D, E, F, G) are listed.
# This prevents duplicate execution of shared ancestor logic.


=== Class: H ===
Custom C3: ['H', 'F', 'D', 'G', 'B', 'E', 'C', 'A', 'object']
Native MRO:  ['H', 'F', 'D', 'G', 'B', 'E', 'C', 'A', 'object']
Method 'solve' resolved to: F


## Задание 3

Исследовать, как именно работает name mangling в CPython для «закрытых» атрибутов и как это отражается в `__dict__` и `dir()`.

- Реализуйте класс SecureBase c атрибутами:
  - `__secret_value`
  - `_semi_private`
  - `public`

- Унаследуйте от него класс `SecureChild`, где:
  - Переопределите `__secret_value` и `_semi_private`.
  - Добавьте метод, который возвращает содержимое `self.__dict__`.

- Напишите код, который:
  - Показывает результат `dir()` и `__dict__` для экземпляров обоих классов.
  - Демонстрирует, под какими реальными именами хранятся «закрытые» атрибуты.
  - Пытается получить доступ к «закрытому» атрибуту через сгенерированное имя (`_ИмяКласса__secret_value`).


In [76]:
class SecureBase:
    def __init__(self):
        self.__secret_value = "Base Secret"
        self._semi_private = "Base Semi"
        self.public = "Base Public"


class SecureChild(SecureBase):
    def __init__(self):
        super().__init__()
        # These create new attributes due to name mangling, they do not override base's
        self.__secret_value = "Child Secret"
        self._semi_private = "Child Semi"
        self.public = "Child Public"

    def dump_dict(self):
        return self.__dict__


base = SecureBase()
child = SecureChild()

print("=== 1. dir() Output ===")
print(
    f"Base dir (filtered): {[x for x in dir(base) if not x.startswith('_') or 'secret' in x or 'semi' in x]}"
)
# '_SecureBase__secret_value' appears in dir(base)
print(
    f"Child dir (filtered): {[x for x in dir(child) if not x.startswith('_') or 'secret' in x or 'semi' in x]}"
)
# both '_SecureBase__secret_value' and '_SecureChild__secret_value' appear

print("\n=== 2. __dict__ Contents ===")
print(f"Base __dict__: {base.__dict__}")
print(f"Child __dict__: {child.dump_dict()}")

print("\n=== 3. Access via Mangled Names ===")
print(f"Child accessing Base's secret: {child._SecureBase__secret_value}")
print(f"Child accessing its own secret: {child._SecureChild__secret_value}")
print(
    f"Accessing through generated name: {child.__getattribute__(f'_{base.__class__.__name__}__secret_value')}"
)

=== 1. dir() Output ===
Base dir (filtered): ['_SecureBase__secret_value', '_semi_private', 'public']
Child dir (filtered): ['_SecureBase__secret_value', '_SecureChild__secret_value', '_semi_private', 'dump_dict', 'public']

=== 2. __dict__ Contents ===
Base __dict__: {'_SecureBase__secret_value': 'Base Secret', '_semi_private': 'Base Semi', 'public': 'Base Public'}
Child __dict__: {'_SecureBase__secret_value': 'Base Secret', '_semi_private': 'Child Semi', 'public': 'Child Public', '_SecureChild__secret_value': 'Child Secret'}

=== 3. Access via Mangled Names ===
Child accessing Base's secret: Base Secret
Child accessing its own secret: Child Secret
Accessing through generated name: Base Secret


## Задание 4

Показать влияние `__slots__` на структуру объекта, наличие `__dict__` и возможность динамического добавления атрибутов, а также `weakref`.

- Опишите три класса:
  - NoSlots: без `__slots__`.
  - WithSlots: с `__slots__ = ("x", "y")`.
  - WithSlotsWeak: с `__slots__ = ("x", "__weakref__")`.
- Для каждого класса:
  - Создайте серию экземпляров, замерьте:
    - Наличие `__dict__` и `__weakref__` (через `hasattr` и `dir`).
    - Возможность динамически добавить новый атрибут `z`.
  - Используя модуль sys, оцените примерный размер одного экземпляра (через getsizeof плюс, при наличии, размер `__dict__`).
- Покажите, для каких классов возможно создавать слабые ссылки (`weakref.ref`).

In [77]:
class NoSlots:
    def __init__(self, x, y):
        self.x = x
        self.y = y


class WithSlots:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y


class WithSlotsWeak:
    __slots__ = ("x", "__weakref__")

    def __init__(self, x):
        self.x = x


def describe_instance(obj):
    d = {
        "type": type(obj).__name__,
        "has_dict": hasattr(obj, "__dict__"),
        "has_weakref": hasattr(obj, "__weakref__"),
        "size_obj": sys.getsizeof(obj),
    }
    if hasattr(obj, "__dict__"):
        d["size_dict"] = sys.getsizeof(obj.__dict__)
        d["dict_keys"] = list(obj.__dict__.keys())
    else:
        d["size_dict"] = 0
        d["dict_keys"] = []
    return d


n = NoSlots(1, 2)
w = WithSlots(1, 2)
ww = WithSlotsWeak(1)

instances = [n, w, ww]
results = []

for obj in instances:
    info = describe_instance(obj)

    # attribute test
    can_add_z = False
    try:
        obj.z = 99
        can_add_z = True
        if hasattr(obj, "__dict__") and "z" in obj.__dict__:
            info["dict_keys"].append("z")
            info["size_dict"] = sys.getsizeof(obj.__dict__)
    except AttributeError:
        can_add_z = False

    info["can_add_z"] = can_add_z

    # Weakref test
    can_weak = False
    try:
        ref = weakref.ref(obj)
        can_weak = True
    except TypeError:
        can_weak = False

    info["can_weakref"] = can_weak
    results.append(info)

print(
    f"{'Class':<15} | {'Dict?':<5} | {'Weak?':<5} | {'Add Z?':<6} | {'Obj Size':<8} | {'Dict Size':<9} | Keys"
)
print("-" * 80)
for r in results:
    keys_str = str(r["dict_keys"]) if r["dict_keys"] else "N/A"
    print(
        f"{r['type']:<15} | {str(r['has_dict']):<5} | {str(r['can_weakref']):<5} | {str(r['can_add_z']):<5} | {r['size_obj']:<5} | {r['size_dict']:<5} | {keys_str}"
    )

Class           | Dict? | Weak? | Add Z? | Obj Size | Dict Size | Keys
--------------------------------------------------------------------------------
NoSlots         | True  | True  | True  | 48    | 296   | ['x', 'y', 'z']
WithSlots       | False | False | False | 48    | 0     | N/A
WithSlotsWeak   | False | True  | False | 56    | 0     | N/A


## Задание 5

Исследовать, как слабые ссылки учитываются в подсчёте ссылок и как ведут себя при циклических структурах.

- Определите класс Node, который:
  - Может ссылаться на «родителя» через обычную сильную ссылку.
  - Может ссылаться на «родителя» через weakref.ref.
- Постройте:
  - Циклический граф с сильными ссылками и измерьте:
  - Счётчики ссылок через sys.getrefcount для ключевых объектов.
  - Поведение GC до и после удаления внешних ссылок (модуль gc).
- Аналогичную структуру, но часть ссылок сделайте слабыми.

- Покажите:
  - Что происходит с результатом вызова слабой ссылки после удаления объекта.
  - Как GC обрабатывает циклы со слабыми и без слабых ссылок.

In [78]:
class Node:
    def __init__(self, name, parent=None, weak_parent=False):
        self.name = name
        if weak_parent and parent is not None:
            self._parent_ref = weakref.ref(parent)
        else:
            self._parent_ref = parent

    def get_parent(self):
        if isinstance(self._parent_ref, weakref.ref):
            return self._parent_ref()
        return self._parent_ref


def refcount(obj):
    return sys.getrefcount(obj)


print("=== Strong Reference Cycle ===")
a = Node("A")
b = Node("B", parent=a)
a._parent_ref = b  # Create cycle: A -> B -> A

print(f"Refcount A: {refcount(a)}")
print(f"Refcount B: {refcount(b)}")

del a
print(gc.collect())  # two items

print(f"B's parent after A dies: {b.get_parent()}")

print("\n=== Weak Reference Cycle ===")
c = Node("C")
d = Node("D", parent=c, weak_parent=True)
c._parent_ref = weakref.ref(d)

print(f"Refcount C: {refcount(c)}")
print(f"Refcount D: {refcount(d)}")
wr_c = weakref.ref(c)
print(f"wr_c before del: {wr_c() is not None}")

del c
print(gc.collect())  # 7 items

print(f"wr_c after del: {wr_c()}")
print(f"D's parent after C dies: {d.get_parent()}")

=== Strong Reference Cycle ===
Refcount A: 3
Refcount B: 3
80
B's parent after A dies: <__main__.Node object at 0x0000024FFADF17F0>

=== Weak Reference Cycle ===
Refcount C: 2
Refcount D: 2
wr_c before del: True
7
wr_c after del: None
D's parent after C dies: None
